# 🚀 머신러닝 실습 : 산탄데르 은행 데이터로 고객 만족 예측 모델링 (분류 문제)

* 주어진 데이터세트는 산탄데르 은행의 익명화된 데이터입니다
* 모델의 성능은 자유롭게 측정해봅니다!


<br/>

---

<br/>
<br/>

# 0. 라이브러리 불러오기

* 라이브러리를 가져와서 과정을 준비합니다

In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings(action='ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier


<br/>

---

<br/>
<br/>

# 1. 데이터 불러오기
* 데이터를 가져와서 과정을 준비합시다.
- 데이터 출처 : https://www.kaggle.com/competitions/santander-customer-satisfaction

In [1]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: C:\githome\hipython_rep


In [3]:
df = pd.read_csv("./data1/train.csv")

<br/>

---

<br/>
<br/>

# 2. 데이터 탐색하기
* 데이터를 이해할 수 있도록 탐색과정을 수행해봅시다.

데이터의 상위 몇 개 행을 출력하여 전체 구조를 미리 확인합니다.

In [4]:
df.head()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [11]:
df.describe()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


<br/>

---

<br/>
<br/>

# 3. 데이터 전처리
* 전처리 과정을 통해서 머신러닝에 사용할 수 있는 형태의 데이터 준비

In [13]:
X = df
X = X.drop('ID', axis=1)
X = X.drop('TARGET', axis=1)
y = df[['TARGET']]

각 데이터에 표준화를 적용하여 데이터의 스케일(크기 차이)을 맞춰줍니다.
- 평균을 0, 표준편차를 1로 맞춰서 → 데이터가 정규 분포 형태로 변환되도록 하세요

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

In [24]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

<br/>

---

<br/>
<br/>

# 5-1. 모델링 - LogisticRegression

* 본격적으로 모델을 선언하고 학습시킵니다.

- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [25]:
lr_clf = LogisticRegression()
lr_clf.fit(X_train_scaled, y_train)
lr_pred = lr_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [26]:
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

[[14597    14]
 [  591     2]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     14611
           1       0.12      0.00      0.01       593

    accuracy                           0.96     15204
   macro avg       0.54      0.50      0.49     15204
weighted avg       0.93      0.96      0.94     15204




<br/>

---

<br/>
<br/>

# 5-2. 모델링 - DecisionTreeClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.

- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [36]:
dt_clf = DecisionTreeClassifier()
dt_clf.fit(X_train_scaled, y_train)
dt_pred = dt_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [37]:
print(confusion_matrix(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

[[14062   549]
 [  510    83]]
              precision    recall  f1-score   support

           0       0.97      0.96      0.96     14611
           1       0.13      0.14      0.14       593

    accuracy                           0.93     15204
   macro avg       0.55      0.55      0.55     15204
weighted avg       0.93      0.93      0.93     15204




<br/>

---

<br/>
<br/>

# 5-3. 모델링 - RandomForestClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.


- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [32]:
rf_clf = RandomForestClassifier(random_state=55, max_depth=8)
rf_clf.fit(X_train_scaled,y_train)
rf_pred = rf_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [33]:
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

[[14611     0]
 [  593     0]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     14611
           1       0.00      0.00      0.00       593

    accuracy                           0.96     15204
   macro avg       0.48      0.50      0.49     15204
weighted avg       0.92      0.96      0.94     15204




<br/>

---

<br/>
<br/>

# 5-4. 모델링 - XGBoost

* 본격적으로 모델을 선언하고 학습시킵니다.


- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [34]:
xgb = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3, use_label_encoder=False)
evals = [(X_test_scaled, y_test)]
xgb.fit(X_train_scaled, y_train, early_stopping_rounds=40, 
        eval_set=evals, verbose=True)
xgb_pred = xgb.predict(X_test_scaled)

[0]	validation_0-logloss:0.61146
[1]	validation_0-logloss:0.54463
[2]	validation_0-logloss:0.48907
[3]	validation_0-logloss:0.44231
[4]	validation_0-logloss:0.40260
[5]	validation_0-logloss:0.36859
[6]	validation_0-logloss:0.33929
[7]	validation_0-logloss:0.31399
[8]	validation_0-logloss:0.29203
[9]	validation_0-logloss:0.27287
[10]	validation_0-logloss:0.25613
[11]	validation_0-logloss:0.24153
[12]	validation_0-logloss:0.22874
[13]	validation_0-logloss:0.21751
[14]	validation_0-logloss:0.20763
[15]	validation_0-logloss:0.19896
[16]	validation_0-logloss:0.19131
[17]	validation_0-logloss:0.18454
[18]	validation_0-logloss:0.17856
[19]	validation_0-logloss:0.17335
[20]	validation_0-logloss:0.16874
[21]	validation_0-logloss:0.16464
[22]	validation_0-logloss:0.16104
[23]	validation_0-logloss:0.15791
[24]	validation_0-logloss:0.15512
[25]	validation_0-logloss:0.15268
[26]	validation_0-logloss:0.15051
[27]	validation_0-logloss:0.14860
[28]	validation_0-logloss:0.14687
[29]	validation_0-loglos

In [35]:
print(confusion_matrix(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))

[[14609     2]
 [  590     3]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     14611
           1       0.60      0.01      0.01       593

    accuracy                           0.96     15204
   macro avg       0.78      0.50      0.50     15204
weighted avg       0.95      0.96      0.94     15204



<br/>

---


<br/>

## 7.  위 4가지 모델의 학습 & 예측 & 평가 결과를 확인하고 최고 성능을 내는 모델을 찾아봅시다!

- 어떤 모델이 가장 성능이 좋은가요 ?